# RAG Evaluation with Ragas + Langfuse

This notebook demonstrates how to evaluate Retrieval-Augmented Generation (RAG) pipelines using [Ragas](https://docs.ragas.io/) metrics and [Langfuse](https://langfuse.com/) for tracing and score logging.

Source: https://langfuse.com/guides/cookbook/evaluation_of_rag_with_ragas

**Two evaluation approaches covered:**
1. **Single trace scoring** — score a trace in real-time as it runs
2. **Batch scoring** — periodically retrieve past traces and score them in bulk

**Ragas metrics used:**
- `Faithfulness` — are the answers grounded in the retrieved context?
- `ResponseRelevancy` — is the response relevant to the question?
- `LLMContextPrecisionWithoutReference` — is the retrieved context precise?

## 1. Install Dependencies

In [ ]:
%pip install langfuse datasets ragas llama_index python-dotenv openai langchain-openai --upgrade

## 2. Configure Credentials

Set your Langfuse and OpenAI credentials. You can find your Langfuse keys at https://cloud.langfuse.com under Project Settings.

Regional endpoints:
- EU: `https://cloud.langfuse.com`
- US: `https://us.cloud.langfuse.com`
- Japan: `https://jp.cloud.langfuse.com`
- HIPAA: `https://hipaa.cloud.langfuse.com`

In [ ]:
import os

os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"
os.environ["OPENAI_API_KEY"] = "sk-proj-..."

## 3. Load Evaluation Dataset

We use the [FIQA](https://huggingface.co/datasets/explodinggradients/fiqa) dataset from Hugging Face. It contains questions, answers, contexts, and ground truths — a convenient stand-in for a real RAG pipeline's outputs.

In [ ]:
from datasets import load_dataset

fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")["baseline"]
fiqa_eval

## 4. Configure Ragas Metrics

Initialize the three metrics we will use throughout this notebook.

In [ ]:
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
)

metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    LLMContextPrecisionWithoutReference(),
]

### Initialize Metrics with LLM + Embeddings

Ragas metrics need an LLM and embedding model to evaluate. We wrap LangChain's OpenAI clients with Ragas adapters.

In [ ]:
from ragas.run_config import RunConfig
from ragas.metrics.base import MetricWithLLM, MetricWithEmbeddings

def init_ragas_metrics(metrics, llm, embedding):
    for metric in metrics:
        if isinstance(metric, MetricWithLLM):
            metric.llm = llm
        if isinstance(metric, MetricWithEmbeddings):
            metric.embeddings = embedding
        run_config = RunConfig()
        metric.init(run_config)

In [ ]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

llm = ChatOpenAI()
emb = OpenAIEmbeddings()

init_ragas_metrics(
    metrics,
    llm=LangchainLLMWrapper(llm),
    embedding=LangchainEmbeddingsWrapper(emb),
)

## 5. Initialize Langfuse Client

In [ ]:
from langfuse import get_client

langfuse = get_client()

In [ ]:
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

---

## Approach 1: Single Trace Scoring

Score a trace in real-time — useful for **online evaluation** during development or for flagging low-quality responses immediately.

The flow:
1. Create a Langfuse trace with `retrieval` and `generation` spans
2. Score the trace with Ragas
3. Push scores back to Langfuse attached to that trace

In [ ]:
row = fiqa_eval[0]
row["question"], row["answer"]

In [ ]:
from ragas.dataset_schema import SingleTurnSample

async def score_with_ragas(query, chunks, answer):
    scores = {}
    for m in metrics:
        sample = SingleTurnSample(
            user_input=query,
            retrieved_contexts=chunks,
            response=answer,
        )
        print(f"calculating {m.name}")
        scores[m.name] = await m.single_turn_ascore(sample)
    return scores

In [ ]:
question = row["question"]
contexts = row["contexts"]
answer = row["answer"]

with langfuse.start_as_current_observation(as_type="span", name="rag") as trace:
    trace_id = trace.trace_id

    with trace.start_as_current_observation(
        name="retrieval",
        input={"question": question},
        output={"contexts": contexts},
    ):
        pass

    with trace.start_as_current_observation(
        name="generation",
        input={"question": question, "contexts": contexts},
        output={"answer": answer},
    ):
        pass

    ragas_scores = await score_with_ragas(question, contexts, answer)

print("RAGAS Scores:", ragas_scores)
ragas_scores

### Push Scores to Langfuse

Attach the Ragas scores to the trace we just created so they appear in the Langfuse UI.

In [ ]:
for m in metrics:
    langfuse.create_score(
        name=m.name,
        value=ragas_scores[m.name],
        trace_id=trace_id,
    )

---

## Approach 2: Batch Scoring

Evaluate a sample of past traces in bulk — useful for **periodic offline evaluation** to monitor quality trends without scoring every request in real-time.

The flow:
1. Create demo RAG traces in Langfuse (simulating production traffic)
2. Retrieve a sample of those traces via the Langfuse API
3. Run Ragas batch evaluation
4. Push aggregate scores back to each trace

### 2a. Create Demo Traces (simulate production traffic)

In [ ]:
for interaction in fiqa_eval.select(range(10, 20)):
    question = interaction["question"]
    contexts = interaction["contexts"]
    answer = interaction["answer"]

    with langfuse.start_as_current_observation(as_type="span", name="rag") as trace:
        with trace.start_as_current_observation(
            name="retrieval",
            input={"question": question},
            output={"contexts": contexts},
        ):
            pass

        with trace.start_as_current_observation(
            name="generation",
            input={"question": question, "contexts": contexts},
            output={"answer": answer},
        ):
            pass

langfuse.flush()
print("Flushed 10 demo traces to Langfuse.")

### 2b. Retrieve Traces from Langfuse

In [ ]:
def get_traces(name=None, limit=None, user_id=None):
    all_data = []
    page = 1

    while True:
        response = langfuse.api.trace.list(
            name=name, page=page, user_id=user_id
        )
        if not response.data:
            break
        page += 1
        all_data.extend(response.data)
        if len(all_data) >= limit:
            break

    return all_data[:limit]

In [ ]:
from random import sample

NUM_TRACES_TO_SAMPLE = 3
traces = get_traces(name="rag", limit=5)
traces_sample = sample(traces, NUM_TRACES_TO_SAMPLE)

print(f"Sampled {len(traces_sample)} traces for evaluation.")

### 2c. Build Evaluation Batch

Extract the inputs/outputs from each trace's observations to reconstruct the RAG inputs needed by Ragas.

In [ ]:
evaluation_batch = {
    "question": [],
    "contexts": [],
    "answer": [],
    "trace_id": [],
}

for t in traces_sample:
    observations = [langfuse.api.observations.get(o) for o in t.observations]
    for o in observations:
        if o.name == "retrieval":
            question = o.input["question"]
            contexts = o.output["contexts"]
        if o.name == "generation":
            answer = o.output["answer"]
    evaluation_batch["question"].append(question)
    evaluation_batch["contexts"].append(contexts)
    evaluation_batch["answer"].append(answer)
    evaluation_batch["trace_id"].append(t.id)

print(f"Built evaluation batch with {len(evaluation_batch['question'])} rows.")

### 2d. Run Batch Evaluation with Ragas

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy

ds = Dataset.from_dict(evaluation_batch)
r = evaluate(ds, metrics=[Faithfulness(), ResponseRelevancy()])
r

### 2e. Export Results to Pandas

In [ ]:
df = r.to_pandas()
df["trace_id"] = ds["trace_id"]
df.head()

### 2f. Push Batch Scores Back to Langfuse

Associate each score with its original trace so they're visible in the Langfuse dashboard.

In [ ]:
for _, row in df.iterrows():
    for metric_name in ["faithfulness", "answer_relevancy"]:
        langfuse.create_score(
            name=metric_name,
            value=row[metric_name],
            trace_id=row["trace_id"],
        )

print("Scores pushed to Langfuse.")

---

## Summary

| Approach | When to use | Cost |
|---|---|---|
| **Single trace** | Real-time feedback, dev/staging, flagging bad responses | Higher — scores every request |
| **Batch scoring** | Production monitoring, quality trend analysis | Lower — samples a subset periodically |

Both approaches log scores directly to Langfuse traces, letting you filter, compare, and alert on RAG quality over time without leaving the Langfuse UI.